# 02 — Feature Engineering

**Goal**: Transform raw Sparkov columns into ML-ready features

### New Features
| Feature | Type | Description |
|---------|------|-------------|
| `log_amt` | Numeric | log1p(amount) — handles heavy right skew |
| `log_distance_km` | Numeric | log1p(Haversine distance customer↔merchant) |
| `log_city_pop` | Numeric | log1p(city population) |
| `amt_to_pop_ratio` | Numeric | Suspicious: high spend in small city |
| `hour` | Numeric | Transaction hour (0-23) |
| `day_of_week` | Numeric | 0=Mon … 6=Sun |
| `is_night` | Binary | 1 if 11pm–5am |
| `is_weekend` | Binary | 1 if Sat/Sun |
| `age` | Numeric | Customer age at transaction time |
| `category` | Categorical | Merchant category |

---

In [ ]:
import sys; sys.path.insert(0, '..')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings; warnings.filterwarnings('ignore')

plt.rcParams.update({'figure.figsize': (14, 5), 'figure.dpi': 110,
                     'axes.spines.top': False, 'axes.spines.right': False})
FRAUD_PAL = {0: '#2196F3', 1: '#F44336'}

from src.data.loader import load_train_test
from src.data.features import (
    haversine_km, engineer_features, split_features_target,
    NUMERIC_FEATURES, CATEGORICAL_FEATURES, FEATURE_NAMES
)

df_train_raw, df_test_raw = load_train_test(
    '../data/fraudTrain.csv', '../data/fraudTest.csv', train_sample_size=200_000
)
print('Data loaded ✅')

## Step 1: Apply Feature Engineering

In [ ]:
df_train = engineer_features(df_train_raw)
df_test  = engineer_features(df_test_raw)

print(f'Columns before: {df_train_raw.shape[1]} → after: {df_train.shape[1]}')
print(f'New numeric features: {NUMERIC_FEATURES}')
print(f'Categorical features: {CATEGORICAL_FEATURES}')
df_train[NUMERIC_FEATURES].describe().round(3)

## Step 2: Validate log_amt Transform

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, col, title in zip(axes, ['amt', 'log_amt'],
                            ['Raw Transaction Amount', 'log1p(Amount)']):
    src = df_train_raw if col == 'amt' else df_train
    for label, color in FRAUD_PAL.items():
        subset = src[src['is_fraud']==label][col] if col=='amt' else src[src['is_fraud']==label][col]
        ax.hist(subset.clip(0, subset.quantile(0.99)), bins=60,
                alpha=0.6, color=color, density=True, label=['Legit','Fraud'][label])
    ax.set_title(title, fontweight='bold'); ax.legend(fontsize=9)
fig.suptitle('Amount Distribution: Raw vs Log-Transformed', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

## Step 3: Validate Haversine Distance Feature

In [ ]:
from scipy import stats

fraud_dist   = df_train[df_train['is_fraud']==1]['log_distance_km']
legit_dist   = df_train[df_train['is_fraud']==0]['log_distance_km']
t_stat, p_val = stats.ttest_ind(legit_dist, fraud_dist)

print(f'log_distance_km — Legit mean: {legit_dist.mean():.3f} | Fraud mean: {fraud_dist.mean():.3f}')
print(f'T-test: t={t_stat:.2f}, p={p_val:.2e} ← {'highly significant ✅' if p_val < 0.001 else ''}')

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(legit_dist, bins=60, alpha=0.55, color='#2196F3', label='Legitimate', density=True)
ax.hist(fraud_dist, bins=60, alpha=0.55, color='#F44336', label='Fraud', density=True)
ax.axvline(legit_dist.mean(), color='#2196F3', ls='--', lw=2, label=f'Legit mean={legit_dist.mean():.2f}')
ax.axvline(fraud_dist.mean(), color='#F44336', ls='--', lw=2, label=f'Fraud mean={fraud_dist.mean():.2f}')
ax.set_xlabel('log1p(Distance km)')
ax.set_title('Haversine Distance Feature Validation', fontsize=12, fontweight='bold')
ax.legend(); plt.tight_layout(); plt.show()

## Step 4: Temporal Feature Validation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Fraud rate by hour
hour_fraud = df_train.groupby('hour')['is_fraud'].mean() * 100
night_mask = list(range(23, 24)) + list(range(0, 6))
colors_hour = ['#F44336' if h in night_mask else '#2196F3' for h in hour_fraud.index]
axes[0].bar(hour_fraud.index, hour_fraud.values, color=colors_hour, alpha=0.8)
axes[0].set_xlabel('Hour of Day'); axes[0].set_ylabel('Fraud Rate (%)')
axes[0].set_title('Fraud Rate by Hour\n(Red = is_night=1)', fontweight='bold')

# is_night validation
night_fraud  = df_train[df_train['is_night']==1]['is_fraud'].mean() * 100
day_fraud    = df_train[df_train['is_night']==0]['is_fraud'].mean() * 100
weekend_fraud  = df_train[df_train['is_weekend']==1]['is_fraud'].mean() * 100
weekday_fraud  = df_train[df_train['is_weekend']==0]['is_fraud'].mean() * 100

binary_feats = ['Night\n(is_night=1)', 'Day\n(is_night=0)',
                'Weekend\n(is_weekend=1)', 'Weekday\n(is_weekend=0)']
binary_rates = [night_fraud, day_fraud, weekend_fraud, weekday_fraud]
axes[1].bar(binary_feats, binary_rates,
            color=['#F44336','#2196F3','#FF9800','#4CAF50'], alpha=0.85)
axes[1].set_ylabel('Fraud Rate (%)')
axes[1].set_title('Fraud Rate: Binary Time Features', fontweight='bold')
for i, v in enumerate(binary_rates):
    axes[1].text(i, v+0.02, f'{v:.3f}%', ha='center', fontsize=10)

plt.suptitle('Temporal Feature Validation', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

## Step 5: Feature Correlation Matrix

In [ ]:
corr = df_train[NUMERIC_FEATURES + ['is_fraud']].corr()

fig, ax = plt.subplots(figsize=(12, 9))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlBu_r',
            center=0, square=True, linewidths=0.5, ax=ax)
ax.set_title('Feature Correlation Matrix (incl. is_fraud)', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

print('\nCorrelations with is_fraud:')
print(corr['is_fraud'].drop('is_fraud').abs().sort_values(ascending=False).round(3).to_string())

## Step 6: Final Feature Matrix

In [ ]:
X_train, y_train = split_features_target(df_train)
X_test,  y_test  = split_features_target(df_test)

print(f'Feature matrix: {X_train.shape}')
print(f'Fraud in train: {y_train.sum():,} ({y_train.mean():.4%})')
print(f'Fraud in test:  {y_test.sum():,}  ({y_test.mean():.4%})')
print(f'\nAll features ({len(FEATURE_NAMES)}): {FEATURE_NAMES}')

## Summary

| Feature | Discriminative Power | Rationale |
|---------|---------------------|----------|
| `log_distance_km` | ⭐⭐⭐ | Fraud transactions are geographically distant |
| `log_amt` | ⭐⭐ | Fraud tends toward higher amounts |
| `amt_to_pop_ratio` | ⭐⭐ | High spend in small city = suspicious |
| `is_night` | ⭐⭐ | Night transactions have elevated fraud rate |
| `hour` | ⭐⭐ | Non-linear time effect captured |
| `category` | ⭐⭐ | Strong categorical predictor |
| `age` | ⭐ | Modest effect |
| `gender` | ✗ | Minimal predictive power |

> **Next**: → `03_Imbalance_Strategies.ipynb`